In [ ]:
import gc
import os
import psutil
from PIL import Image

# Parent directory containing class folders
parent_dir = 'D:/GCN/Brain_Tumor/four_class'

def find_and_fix_images(directory):
    corrupted_count = 0
    fixed_count = 0
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if file_path.lower().endswith(('.jpeg', '.jpg', '.jpeg', '.png' '.bmp')):
            try:
                # Open the image
                with Image.open(file_path) as img:
                    # Check for alpha channel
                    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
                        print(f"Fixing alpha channel: {file_path}")
                        img = img.convert('RGB')  # Convert to RGB
                        img.save(file_path)  # Save back without alpha channel
                        fixed_count += 1
                    # Verify image integrity
                    img.verify()
            except (IOError, SyntaxError) as e:
                print(f"Removing corrupted file: {file_path}")
                os.remove(file_path)
                corrupted_count += 1
    return corrupted_count, fixed_count

def process_all_subdirectories(parent_directory):

    total_corrupted = 0
    total_fixed = 0
    for sub_dir in os.listdir(parent_directory):
        sub_dir_path = os.path.join(parent_directory, sub_dir)
        if os.path.isdir(sub_dir_path):  # Ensure it's a directory
            print(f"Processing images in: {sub_dir_path}")
            corrupted_in_dir, fixed_in_dir = find_and_fix_images(sub_dir_path)
            print(f"Corrupted files removed from {sub_dir_path}: {corrupted_in_dir}")
            print(f"Images fixed (alpha channel removed) in {sub_dir_path}: {fixed_in_dir}")
            total_corrupted += corrupted_in_dir
            total_fixed += fixed_in_dir
    print(f"Total corrupted files removed: {total_corrupted}")
    print(f"Total images fixed (alpha channel removed): {total_fixed}")

# Start processing
process_all_subdirectories(parent_dir)



***
<a name='import Packages'>
    
# 1 <span style='color:blue'>|</span> Hybrid based KNN graph building model

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    log_loss
)
from sklearn.manifold import TSNE

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter
# ======================
# CONFIG
# ======================
data_dir = r"D:/GCN/Brain_Tumor/four_class"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

AUGMENT = True
LR_CNN = 1e-3
EPOCHS_CNN = 30

K_DEFAULT = 12
USE_PCA = True
PCA_DIM = 256

LR_GCN = 0.005
EPOCHS_GCN = 200
DROPOUT = 0.3

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ======================
# LOAD DATA
# ======================
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels = [], []

for c in class_names:
    for p in glob.glob(os.path.join(data_dir, c, "*")):
        if p.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels.append(class_to_idx[c])

paths = np.array(paths)
labels = np.array(labels, dtype=np.int32)

N = len(paths)
num_classes = len(class_names)

print("Images:", N)
print("Classes:", num_classes)
print("Class names:", class_names)


# ======================
# SPLIT: 70 TRAIN, 20 VAL, 10 TEST
# ======================
idx = np.arange(N)

idx_temp, idx_te = train_test_split(
    idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

idx_tr, idx_va = train_test_split(
    idx_temp,
    test_size=0.2222,
    random_state=SEED,
    stratify=labels[idx_temp]
)

mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)

mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

print("Train:", len(idx_tr))
print("Validation:", len(idx_va))
print("Test:", len(idx_te))


# ======================
# DATA PIPELINE
# ======================
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label


def augment_img(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label


def make_ds(idxs, training=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[idxs], labels[idxs]))

    if shuffle:
        ds = ds.shuffle(len(idxs), seed=SEED)

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.map(augment_img, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_ds(idx_tr, training=True, shuffle=True)
val_ds = make_ds(idx_va, training=False, shuffle=False)
all_ds = make_ds(idx, training=False, shuffle=False)

# ======================
# CNN + PCA + GCN Full Pipeline with Ablation
# ======================

# ----------------------
# 1️⃣ CNN FEATURE EXTRACTOR
# ----------------------
def build_cnn():
    inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    feat = layers.Dense(512, activation="relu", name="feat")(x)
    out = layers.Dropout(0.5)(feat)
    out = layers.Dense(num_classes, activation="softmax")(out)

    cnn_model = Model(inp, out)
    backbone = Model(inp, feat)
    return cnn_model, backbone


cnn, backbone = build_cnn()
cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Optional: Compute class weights
from sklearn.utils.class_weight import compute_class_weight
class_weights_dict = dict(enumerate(
    compute_class_weight("balanced", classes=np.arange(num_classes), y=labels[idx_tr])
))

# Train CNN
cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    class_weight=class_weights_dict,
    callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
    verbose=1
)

# ----------------------
# 2️⃣ EXTRACT CNN FEATURES
# ----------------------
def extract_features(ds):
    X, Y = [], []
    for xb, yb in ds:
        feat = backbone(xb, training=False).numpy()
        X.append(feat)
        Y.append(yb.numpy())
    return np.vstack(X), np.concatenate(Y)

X_all, y_all = extract_features(all_ds)

# ----------------------
# 3️⃣ PCA + STANDARDIZATION
# ----------------------
X_tr = X_all[idx_tr]
X_va = X_all[idx_va]
X_te = X_all[idx_te]

if USE_PCA:
    pca_dim = min(PCA_DIM, X_tr.shape[1])
    pca = PCA(n_components=pca_dim, random_state=SEED)
    X_tr = pca.fit_transform(X_tr)
    X_va = pca.transform(X_va)
    X_te = pca.transform(X_te)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_va = scaler.transform(X_va)
X_te = scaler.transform(X_te)

# Full standardized feature matrix for graph
F = X_tr.shape[1]
X_std = np.zeros((N, F), dtype=np.float32)
X_std[idx_tr] = X_tr
X_std[idx_va] = X_va
X_std[idx_te] = X_te

# ----------------------
# 4️⃣ BUILD DOMAIN GRAPH
# ----------------------
def build_graph_domain_rules(paths_list, class_names):
    N_local = len(paths_list)
    class_to_indices = {c: [] for c in class_names}
    for i, p in enumerate(paths_list):
        cls = os.path.basename(os.path.dirname(p))
        if cls in class_to_indices:
            class_to_indices[cls].append(i)
    rows, cols, data = [], [], []
    for idxs in class_to_indices.values():
        for i in idxs:
            for j in idxs:
                rows.append(i)
                cols.append(j)
                data.append(1.0)
    A = coo_matrix((data, (rows, cols)), shape=(N_local, N_local), dtype=np.float32).tocsr()
    A.setdiag(1.0)
    return A

A = build_graph_domain_rules(paths, class_names)
A_norm = gcn_filter(A)

# ----------------------
# 5️⃣ GCN MODEL
# ----------------------
def build_gcn(F):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N, N), sparse=True)
    h = GCNConv(64, activation="relu", kernel_regularizer=regularizers.l2(5e-4))([X_in, A_in])
    h = layers.Dropout(DROPOUT)(h)
    out = GCNConv(num_classes, activation="softmax")([h, A_in])
    model = Model([X_in, A_in], out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model

Y_cat = to_categorical(labels, num_classes).astype(np.float32)
gcn = build_gcn(F)

# ----------------------
# 6️⃣ ABLATION CONFIGS
# ----------------------
ablation_configs = [
    {"name": "CNN only", "cnn_only": True, "pca": False, "gcn": False, "weighted": False},
    {"name": "CNN + PCA", "cnn_only": True, "pca": True, "gcn": False, "weighted": False},
    {"name": "GCN raw", "cnn_only": False, "pca": False, "gcn": True, "weighted": True},
    {"name": "GCN + PCA", "cnn_only": False, "pca": True, "gcn": True, "weighted": True},
    {"name": "GCN + PCA no weight", "cnn_only": False, "pca": True, "gcn": True, "weighted": False},
]

import pandas as pd
from sklearn.metrics import f1_score

results = []

for cfg in ablation_configs:
    print(f"\n===== Ablation: {cfg['name']} =====")
    
    # CNN only predictions
    if cfg["cnn_only"]:
        preds = np.argmax(cnn.predict(all_ds), axis=1)
        acc = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average="macro")
        print(f"{cfg['name']} - Accuracy: {acc:.4f}, Macro-F1: {macro_f1:.4f}")
        results.append({"Ablation": cfg["name"], "Accuracy": acc, "Macro-F1": macro_f1})
        continue
    
    # GCN training
    gcn_model = build_gcn(F)
    
    # Sample weights
    if cfg["weighted"]:
        freqs = np.bincount(labels[idx_tr])
        inv_freq = 1.0 / freqs
        weights_nodes = np.array([inv_freq[l] for l in labels])
        weights_nodes *= mask_tr
    else:
        weights_nodes = mask_tr.astype(np.float32)
    
    gcn_model.fit(
        [X_std, A_norm],
        Y_cat,
        sample_weight=weights_nodes.astype(np.float32),
        validation_data=([X_std, A_norm], Y_cat, mask_va.astype(np.float32)),
        epochs=EPOCHS_GCN,
        batch_size=N,
        shuffle=False,
        verbose=0
    )
    
    gcn_pred_prob = gcn_model.predict([X_std, A_norm], batch_size=N)
    gcn_pred = np.argmax(gcn_pred_prob, axis=1)
    
    acc = accuracy_score(labels, gcn_pred)
    macro_f1 = f1_score(labels, gcn_pred, average="macro")
    print(f"{cfg['name']} - Accuracy: {acc:.4f}, Macro-F1: {macro_f1:.4f}")
    results.append({"Ablation": cfg["name"], "Accuracy": acc, "Macro-F1": macro_f1})

# ----------------------
# 7️⃣ SUMMARY TABLE & PLOT
# ----------------------
df_results = pd.DataFrame(results)
print("\n===== Ablation Summary =====")
print(df_results)

plt.figure(figsize=(7,4))
sns.barplot(x="Ablation", y="Accuracy", data=df_results, palette="viridis")
plt.title("Ablation Study - Accuracy")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,4))
sns.barplot(x="Ablation", y="Macro-F1", data=df_results, palette="magma")
plt.title("Ablation Study - Macro-F1")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()